# Phase 4A - Setup & Data/Models Reload

In [1]:
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))
print("Torch version:", torch.__version__)
print("Torch's built-in CUDA version:", torch.version.cuda)

CUDA available: True
Device: Tesla T4
Torch version: 2.10.0+cu128
Torch's built-in CUDA version: 12.8


In [2]:
!pip install torch_geometric -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 16.4 MB/s eta 0:00:0000:010:01


In [3]:
!pip install rdkit --no-deps

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 38.1/38.1 MB 52.6 MB/s eta 0:00:00:00:0100:01


In [4]:
!pip install transformers PyTDC captum -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.2/154.2 kB 3.5 MB/s eta 0:00:0000:01
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 151.3/151.3 kB 8.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 151.2/151.2 kB 5.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 151.2/151.2 kB 6.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 151.1/151.1 kB 7.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 151.1/151.1 kB 6.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 151.1/151.1 kB 7.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 151.2/151.2 kB 7.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━

In [5]:
import rdkit
import torch_geometric
import captum
import transformers
from tdc.single_pred import Tox

print("rdkit:", rdkit.__version__)
print("torch_geometric:", torch_geometric.__version__)
print("captum:", captum.__version__)
print("transformers:", transformers.__version__)

rdkit: 2026.03.6
torch_geometric: 2.8.0.post1
captum: 0.9.0
transformers: 5.0.0


In [6]:
from tdc.single_pred import Tox
from rdkit import Chem
from rdkit.Chem.SaltRemover import SaltRemover

data = Tox(name='AMES')
df = data.get_data(format='df')
split = data.get_split(method="scaffold", seed=42, frac=[0.7, 0.1, 0.2])
train_df, valid_df, test_df = split['train'], split['valid'], split['test']

# Standardize SMILES strings (salt stripping + canonicalization)
remover = SaltRemover()

def standardize_smiles(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    mol = remover.StripMol(mol, dontRemoveEverything=True)
    if mol is None or mol.GetNumAtoms() == 0:
        return None
    try:
        Chem.SanitizeMol(mol)
    except Exception:
        return None
    return Chem.MolToSmiles(mol, canonical=True)

df['Drug_standardized'] = df['Drug'].apply(standardize_smiles)
df_clean = df[df['Drug_standardized'].notna()].copy()

# Map original split assignments back using Drug_IDs (to filter)
train_ids = set(train_df['Drug_ID'])
valid_ids = set(valid_df['Drug_ID'])
test_ids = set(test_df['Drug_ID'])

df_clean['split'] = df_clean['Drug_ID'].apply(
    lambda i: 'train' if i in train_ids else ('valid' if i in valid_ids else ('test' if i in test_ids else 'unknown'))
)

print(df_clean['split'].value_counts())

# Ensure no molecules slipped through without a split assignment
assert (df_clean['split'] == 'unknown').sum() == 0, "Some molecules lost their split membership -- check filtering."

Downloading...
100%|██████████| 344k/344k [00:00<00:00, 3.45MiB/s]
Loading...
Done!
100%|██████████| 7278/7278 [00:01<00:00, 4393.31it/s]


split
train    5094
test     1457
valid     727
Name: count, dtype: int64


In [7]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from sklearn.metrics import roc_auc_score, average_precision_score, accuracy_score

device = "cuda" if torch.cuda.is_available() else "cpu"

CHEMBERTA_PATH = "/kaggle/input/models/minoola33/chemberta-v1-base-ames-mutagenicity/pytorch/default/1"
GIN_PATH = "/kaggle/input/models/minoola33/gin-ames-mutagenicity/pytorch/default/1/gin_ames_final.pt"


# Reload ChemBERTa

from transformers import AutoTokenizer, AutoModelForSequenceClassification

tokenizer = AutoTokenizer.from_pretrained(CHEMBERTA_PATH)
chemberta_model = AutoModelForSequenceClassification.from_pretrained(CHEMBERTA_PATH).to(device)
chemberta_model.eval()

# Verify no legacy warning occurs since this is our fine-tuned checkpoint
print("ChemBERTa reloaded. If any missing/unexpected key warning printed above, STOP and report it.")

# Quick shape sanity check
example = df_clean[df_clean['split'] == 'test']['Drug_standardized'].iloc[0]
tokens = tokenizer(example, return_tensors="pt").to(device)
with torch.no_grad():
    out = chemberta_model(**tokens).logits
print("Sanity check logits shape:", out.shape, "values:", out.cpu().numpy())

# Re-evaluate on the full locked test set
test_df_local = df_clean[df_clean['split'] == 'test']
all_probs, all_labels = [], []
with torch.no_grad():
    for _, row in test_df_local.iterrows():
        tok = tokenizer(row['Drug_standardized'], return_tensors="pt", truncation=True, max_length=128).to(device)
        logits = chemberta_model(**tok).logits
        prob = torch.softmax(logits, dim=1)[0, 1].item()
        all_probs.append(prob)
        all_labels.append(row['Y'])

cb_auroc = roc_auc_score(all_labels, all_probs)
cb_auprc = average_precision_score(all_labels, all_probs)
cb_acc = accuracy_score(all_labels, (np.array(all_probs) >= 0.5).astype(int))
print(f"\nChemBERTa reloaded-checkpoint test scores: AUROC={cb_auroc:.4f}  AUPRC={cb_auprc:.4f}  ACC={cb_acc:.4f}")
print("Compare to recorded: AUROC=0.8068  AUPRC=0.8558  ACC=0.7447 -- should match closely (exact, not just similar).")


# Reload GIN

from torch_geometric.nn import GINConv, global_add_pool

class GIN(nn.Module):
    def __init__(self, in_dim, hidden_dim=64, num_layers=3, dropout=0.2):
        super().__init__()
        self.convs = nn.ModuleList()
        self.bns = nn.ModuleList()
        for i in range(num_layers):
            mlp = nn.Sequential(
                nn.Linear(in_dim if i == 0 else hidden_dim, hidden_dim),
                nn.ReLU(),
                nn.Linear(hidden_dim, hidden_dim),
            )
            self.convs.append(GINConv(mlp))
            self.bns.append(nn.BatchNorm1d(hidden_dim))
        self.dropout = dropout
        self.classifier = nn.Linear(hidden_dim, 1)

    def forward(self, x, edge_index, batch):
        for conv, bn in zip(self.convs, self.bns):
            x = conv(x, edge_index)
            x = bn(x)
            x = F.relu(x)
            x = F.dropout(x, p=self.dropout, training=self.training)
        x = global_add_pool(x, batch)
        return self.classifier(x).squeeze(-1)

gin_model = GIN(in_dim=6).to(device)
gin_model.load_state_dict(torch.load(GIN_PATH, map_location=device))
gin_model.eval()
print("\nGIN reloaded successfully.")

# Re-evaluate on the full locked test set (reuse Phase 3 featurizer)
from rdkit import Chem
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader

def mol_to_data(smiles, label):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None or mol.GetNumAtoms() == 0:
        return None
    atom_feats = []
    for atom in mol.GetAtoms():
        atom_feats.append([
            atom.GetAtomicNum(), atom.GetDegree(), atom.GetFormalCharge(),
            int(atom.GetHybridization()), int(atom.GetIsAromatic()), atom.GetTotalNumHs(),
        ])
    x = torch.tensor(atom_feats, dtype=torch.float)
    edge_index = []
    for bond in mol.GetBonds():
        i, j = bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()
        edge_index += [[i, j], [j, i]]
    edge_index = torch.tensor(edge_index, dtype=torch.long).t().contiguous() if edge_index else torch.zeros((2, 0), dtype=torch.long)
    return Data(x=x, edge_index=edge_index, y=torch.tensor([label], dtype=torch.float))

test_data = [mol_to_data(r['Drug_standardized'], r['Y']) for _, r in test_df_local.iterrows()]
test_data = [d for d in test_data if d is not None]
test_loader = DataLoader(test_data, batch_size=128, shuffle=False)

all_logits, all_labels_gin = [], []
with torch.no_grad():
    for batch in test_loader:
        batch = batch.to(device)
        logits = gin_model(batch.x, batch.edge_index, batch.batch)
        all_logits.append(logits.cpu())
        all_labels_gin.append(batch.y.cpu())

logits = torch.cat(all_logits).numpy()
labels_gin = torch.cat(all_labels_gin).numpy()
probs_gin = 1 / (1 + np.exp(-logits))

gin_auroc = roc_auc_score(labels_gin, probs_gin)
gin_auprc = average_precision_score(labels_gin, probs_gin)
gin_acc = accuracy_score(labels_gin, (probs_gin >= 0.5).astype(int))
print(f"\nGIN reloaded-checkpoint test scores: AUROC={gin_auroc:.4f}  AUPRC={gin_auprc:.4f}  ACC={gin_acc:.4f}")
print("Compare to recorded: AUROC=0.7845  AUPRC=0.8322  ACC=0.6788 -- should match closely (exact, not just similar).")

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

ChemBERTa reloaded. If any missing/unexpected key warning printed above, STOP and report it.
Sanity check logits shape: torch.Size([1, 2]) values: [[-1.1466241  1.1689568]]

ChemBERTa reloaded-checkpoint test scores: AUROC=0.8068  AUPRC=0.8558  ACC=0.7447
Compare to recorded: AUROC=0.8068  AUPRC=0.8558  ACC=0.7447 -- should match closely (exact, not just similar).

GIN reloaded successfully.

GIN reloaded-checkpoint test scores: AUROC=0.7845  AUPRC=0.8322  ACC=0.6788
Compare to recorded: AUROC=0.7845  AUPRC=0.8322  ACC=0.6788 -- should match closely (exact, not just similar).


# Phase 4B - GIN Attribution Extraction (Captum IG)

In [8]:
def get_gin_atom_attributions(data, n_steps=50):
    """
    data: a single PyG Data object (one molecule, unbatched).
    Returns: (atom_scores, completeness_check).

    Manual Integrated Gradients implementation. Captum's generic wrapper 
    expands the input tensor across interpolated copies in a single batched 
    forward call, assuming batch dimension equals independent examples. 
    That assumption works for images, text, or tabular data, but fails for a 
    GNN where the first dimension of x represents atoms of a single connected 
    graph. This manual loop evaluates the model once per interpolation step 
    using a correctly shaped input, computing the exact same algorithm 
    without tensor batching conflicts.
    """
    data = data.to(device)
    x = data.x.clone()
    baseline = torch.zeros_like(x)  # Zero atom baseline following standard convention
    batch = torch.zeros(x.size(0), dtype=torch.long, device=device)  # Single graph batch vector
    edge_index = data.edge_index

    diff = x - baseline
    total_grads = torch.zeros_like(x)

    for step in range(1, n_steps + 1):
        alpha = step / n_steps
        x_interp = (baseline + alpha * diff).clone().requires_grad_(True)
        output = gin_model(x_interp, edge_index, batch)  # Scalar output for single graph
        grad = torch.autograd.grad(output, x_interp)[0]
        total_grads += grad

    avg_grads = total_grads / n_steps
    attributions = diff * avg_grads
    atom_scores = attributions.sum(dim=1).detach().cpu().numpy()  # Sum across 6 features to get 1 score per atom

    # IG completeness axiom sanity check: sum of attributions should
    # approximately equal the difference between model output on input and baseline.
    with torch.no_grad():
        pred_input = gin_model(x, edge_index, batch).item()
        pred_baseline = gin_model(baseline, edge_index, batch).item()
    expected_diff = pred_input - pred_baseline
    actual_sum = atom_scores.sum()

    return atom_scores, {"expected_diff": expected_diff, "actual_sum": actual_sum,
                          "diff": abs(expected_diff - actual_sum)}

# Sanity check on one molecule before running the full test set
sample_data = test_data[0]
scores, check = get_gin_atom_attributions(sample_data)
print("Per-atom attribution scores:", scores)
print(f"Completeness check - expected diff: {check['expected_diff']:.4f}, "
      f"actual sum: {check['actual_sum']:.4f}, gap: {check['diff']:.4f}")
print("(Gap should be small, such as under 0.05. If it is large, increase n_steps and recheck.)")

# Run over the full test set and align results to Drug_ID for subsequent matching
test_df_ordered = df_clean[df_clean['split'] == 'test'].reset_index(drop=True)
gin_attributions = {}  # Mapping from Drug_ID to atom scores array

failed = 0
for i, (data, (_, row)) in enumerate(zip(test_data, test_df_ordered.iterrows())):
    try:
        scores, _ = get_gin_atom_attributions(data)
        gin_attributions[row['Drug_ID']] = scores
    except Exception as e:
        failed += 1
    if (i + 1) % 100 == 0 or (i + 1) == len(test_data):
        print(f"  Processed {i + 1} / {len(test_data)} molecules "
              f"({len(gin_attributions)} succeeded, {failed} failed)")

print(f"\nExtracted GIN attributions for {len(gin_attributions)} / {len(test_data)} test molecules "
      f"({failed} failed).")

Per-atom attribution scores: [2.2908237 4.919923  1.6008219 4.302677  4.522694  4.6110497 4.7809277
 3.2208138 3.1731794 4.357586  3.0237827 2.6995208 3.0260117 4.3569107
 3.1838145 3.2290006 4.772095  5.300097  5.1805425 3.17767   2.6821706
 2.703795  3.1882143]
Completeness check - expected diff: 85.4943, actual sum: 84.3041, gap: 1.1901
(Gap should be small, such as under 0.05. If it is large, increase n_steps and recheck.)
  Processed 100 / 1457 molecules (100 succeeded, 0 failed)
  Processed 200 / 1457 molecules (200 succeeded, 0 failed)
  Processed 300 / 1457 molecules (300 succeeded, 0 failed)
  Processed 400 / 1457 molecules (400 succeeded, 0 failed)
  Processed 500 / 1457 molecules (500 succeeded, 0 failed)
  Processed 600 / 1457 molecules (600 succeeded, 0 failed)
  Processed 700 / 1457 molecules (700 succeeded, 0 failed)
  Processed 800 / 1457 molecules (800 succeeded, 0 failed)
  Processed 900 / 1457 molecules (900 succeeded, 0 failed)
  Processed 1000 / 1457 molecules (100

In [9]:
# Save attributions for later phases
import pickle
with open("/kaggle/working/gin_attributions.pkl", "wb") as f:
    pickle.dump(gin_attributions, f)
print("Saved to /kaggle/working/gin_attributions.pkl")

Saved to /kaggle/working/gin_attributions.pkl


# Phase 4C - ChemBERTa Attribution Extraction

In [10]:
import torch
from captum.attr import LayerIntegratedGradients

def chemberta_forward(input_ids, attention_mask):
    return chemberta_model(input_ids=input_ids, attention_mask=attention_mask).logits

# Attribute at the embedding layer, which is the standard approach for text models since input_ids are discrete indices that cannot be interpolated directly. 
# Unlike GIN, this fits the normal use case for Captum batching because attention_mask shares the same batch dimension as input_ids.
lig = LayerIntegratedGradients(chemberta_forward, chemberta_model.roberta.embeddings)

def get_chemberta_token_attributions(smiles, target_class=1, n_steps=50, max_length=128):
    """
    Returns: (tokens, token_scores, offsets, convergence_delta)
    - tokens: token strings for human inspection
    - token_scores: one float per token summed over the hidden dimension
    - offsets: start and end character positions per token in the original SMILES string
    - convergence_delta: Captum built-in completeness axiom check
    """
    encoded = tokenizer(smiles, return_tensors="pt", return_offsets_mapping=True,
                        truncation=True, max_length=max_length)
    offsets = encoded.pop("offset_mapping")[0].tolist()
    input_ids = encoded["input_ids"].to(device)
    attention_mask = encoded["attention_mask"].to(device)

    real_len = int(attention_mask.sum().item())

    # Baseline uses padding tokens everywhere except for the structural start and end tokens,
    # following standard text Integrated Gradients conventions.
    baseline_ids = torch.full_like(input_ids, tokenizer.pad_token_id)
    baseline_ids[0, 0] = input_ids[0, 0]
    baseline_ids[0, real_len - 1] = input_ids[0, real_len - 1]

    attributions, delta = lig.attribute(
        inputs=input_ids,
        baselines=baseline_ids,
        additional_forward_args=(attention_mask,),
        target=target_class,
        n_steps=n_steps,
        return_convergence_delta=True,
    )

    token_scores = attributions.sum(dim=-1).squeeze(0).detach().cpu().numpy()
    tokens = tokenizer.convert_ids_to_tokens(input_ids[0].cpu().tolist())

    # Trim to include only real non-padding tokens
    tokens = tokens[:real_len]
    token_scores = token_scores[:real_len]
    offsets = offsets[:real_len]

    return tokens, token_scores, offsets, delta.item()

# Sanity check on one molecule
sample_smiles = df_clean[df_clean['split'] == 'test']['Drug_standardized'].iloc[0]
print("SMILES:", sample_smiles)
print()

tokens, scores, offsets, gap = get_chemberta_token_attributions(sample_smiles)

for tok, sc, off in zip(tokens, scores, offsets):
    print(f"{tok!r:15s}  char_offset={off}  score={sc:.4f}")

print(f"\nConvergence delta (completeness gap): {gap:.4f}")
print("(Judge this relative to the prediction magnitude of the model rather than a fixed "
      "absolute threshold.)")

SMILES: O=[N+]([O-])c1c2c(c3ccc4cccc5ccc1c3c45)CCCC2

'<s>'            char_offset=[0, 0]  score=0.0000
'O'              char_offset=[0, 1]  score=-0.0924
'=['             char_offset=[1, 3]  score=-0.2310
'N'              char_offset=[3, 4]  score=-0.0246
'+](['           char_offset=[4, 8]  score=-0.0268
'O'              char_offset=[8, 9]  score=-0.0679
'-])'            char_offset=[9, 12]  score=0.1847
'c'              char_offset=[12, 13]  score=-0.0553
'1'              char_offset=[13, 14]  score=-0.1291
'c'              char_offset=[14, 15]  score=0.0585
'2'              char_offset=[15, 16]  score=-0.0473
'c'              char_offset=[16, 17]  score=-0.0042
'('              char_offset=[17, 18]  score=0.0260
'c'              char_offset=[18, 19]  score=-0.1525
'3'              char_offset=[19, 20]  score=0.1544
'ccc'            char_offset=[20, 23]  score=0.0604
'4'              char_offset=[23, 24]  score=0.5597
'cccc'           char_offset=[24, 28]  score=0.5224
'5'          

#### Atom Character-span Extraction and Verification

In [11]:
# Cross-references ChemBERTa token offsets against RDKit atom indices.

import re
from rdkit import Chem

SMILES_TOKEN_REGEX = re.compile(
    r"(\[[^\]]+\]|Br|Cl|[BCNOSPFI]|[bcnosp]|\(|\)|\.|=|#|-|\+|\\|/|:|~|@|\?|>|\*|\$|%[0-9]{2}|[0-9])"
)

def get_atom_char_spans(smiles):
    """
    Tokenize a SMILES string into chemical tokens using regex,
    identify atoms from organic subset symbols or bracket atoms,
    and return their character spans in left-to-right order to
    match RDKit atom indexing.

    Returns a list of (start_char, end_char) tuples, one per atom,
    indexed to match mol.GetAtomWithIdx(i).
    """
    atom_spans = []
    for match in SMILES_TOKEN_REGEX.finditer(smiles):
        token = match.group(0)
        start, end = match.span()
        is_atom = (
            token.startswith('[') or
            token in ('Br', 'Cl') or
            token in ('B', 'C', 'N', 'O', 'S', 'P', 'F', 'I') or
            token in ('b', 'c', 'n', 'o', 's', 'p')
        )
        if is_atom:
            atom_spans.append((start, end))
    return atom_spans

# Verify against RDKit atom order for the molecule used in the ChemBERTa sanity check
mol = Chem.MolFromSmiles(sample_smiles)
spans = get_atom_char_spans(sample_smiles)

print(f"RDKit atom count: {mol.GetNumAtoms()}")
print(f"Regex-derived atom span count: {len(spans)}")
assert mol.GetNumAtoms() == len(spans), \
    "Regex tokenizer disagrees with RDKit atom count. Stop and report this before continuing."

print("\nAtom-by-atom comparison (symbol should visually match the character span):")
for i, (s, e) in enumerate(spans):
    atom = mol.GetAtomWithIdx(i)
    print(f"  Atom {i:2d}: RDKit symbol={atom.GetSymbol():3s}  "
          f"chars[{s}:{e}]='{sample_smiles[s:e]}'")

RDKit atom count: 23
Regex-derived atom span count: 23

Atom-by-atom comparison (symbol should visually match the character span):
  Atom  0: RDKit symbol=O    chars[0:1]='O'
  Atom  1: RDKit symbol=N    chars[2:6]='[N+]'
  Atom  2: RDKit symbol=O    chars[7:11]='[O-]'
  Atom  3: RDKit symbol=C    chars[12:13]='c'
  Atom  4: RDKit symbol=C    chars[14:15]='c'
  Atom  5: RDKit symbol=C    chars[16:17]='c'
  Atom  6: RDKit symbol=C    chars[18:19]='c'
  Atom  7: RDKit symbol=C    chars[20:21]='c'
  Atom  8: RDKit symbol=C    chars[21:22]='c'
  Atom  9: RDKit symbol=C    chars[22:23]='c'
  Atom 10: RDKit symbol=C    chars[24:25]='c'
  Atom 11: RDKit symbol=C    chars[25:26]='c'
  Atom 12: RDKit symbol=C    chars[26:27]='c'
  Atom 13: RDKit symbol=C    chars[27:28]='c'
  Atom 14: RDKit symbol=C    chars[29:30]='c'
  Atom 15: RDKit symbol=C    chars[30:31]='c'
  Atom 16: RDKit symbol=C    chars[31:32]='c'
  Atom 17: RDKit symbol=C    chars[33:34]='c'
  Atom 18: RDKit symbol=C    chars[35:36

This signals a real complexity where tokens can also straddle the boundary between an atom and pure syntax, or even between two different atoms., which calls for a more principled approach than token-level splitting: redistribute at the character level instead.

In [12]:
# Token-to-atom alignment (character-level redistribution)

def align_tokens_to_atoms(tokens, token_scores, token_offsets, atom_spans):
    """
    Redistributes token-level attribution to atom-level via character-level
    uniform redistribution:
      1. Each token score is spread uniformly across the characters it spans.
      2. Each atom score equals the sum of character-level mass within its span.
      3. Mass falling outside all atom spans, such as syntax characters or 
         special tokens with no span, is tracked separately as unassigned mass.

    Important: the unassigned fraction is computed using magnitude absolute value 
    accounting rather than signed sums. Using the signed total as a denominator 
    is numerically unstable because positive and negative token contributions 
    can nearly cancel out, which is common for molecules near the decision boundary 
    of the model. This makes the denominator close to zero and causes percentages 
    to blow up arbitrarily. Magnitude accounting treats every contribution as a positive 
    weight, ensuring the denominator is never near zero and the fraction remains stable.

    The actual per-atom scores returned remain signed, as they are needed downstream 
    for Phase 5 faithfulness tests. Only the unassigned fraction metric uses magnitude.
    """
    special_token_mass_signed = 0.0
    special_token_mass_abs = 0.0
    max_char = max(end for (_, end) in atom_spans)
    max_char = max(max_char, max((end for (_, end) in token_offsets), default=0))
    char_mass_signed = np.zeros(max_char)
    char_mass_abs = np.zeros(max_char)

    for tok, score, (start, end) in zip(tokens, token_scores, token_offsets):
        if end <= start:  # Special tokens like <s> or </s> lack real character spans
            special_token_mass_signed += score
            special_token_mass_abs += abs(score)
            continue
        length = end - start
        char_mass_signed[start:end] += score / length
        char_mass_abs[start:end] += abs(score) / length

    atom_scores = np.zeros(len(atom_spans))        # Signed scores for downstream use
    atom_scores_abs = np.zeros(len(atom_spans))    # Magnitude scores for fraction accounting only
    covered = np.zeros(max_char, dtype=bool)
    for i, (start, end) in enumerate(atom_spans):
        atom_scores[i] = char_mass_signed[start:end].sum()
        atom_scores_abs[i] = char_mass_abs[start:end].sum()
        covered[start:end] = True

    syntax_mass_abs = char_mass_abs[~covered].sum()
    unassigned_mass_abs = special_token_mass_abs + syntax_mass_abs
    total_mass_abs = char_mass_abs.sum() + special_token_mass_abs

    # Internal consistency check on magnitude accounting that must hold exactly
    assert abs((atom_scores_abs.sum() + unassigned_mass_abs) - total_mass_abs) < 1e-6, \
        "Magnitude accounting does not add up. There is a bug in the redistribution logic."

    unassigned_fraction = (unassigned_mass_abs / total_mass_abs
                           if total_mass_abs > 1e-8 else float('nan'))

    return {
        "atom_scores": atom_scores,                # Signed scores for Phase 5
        "unassigned_mass_abs": unassigned_mass_abs,
        "special_token_mass_abs": special_token_mass_abs,
        "syntax_mass_abs": syntax_mass_abs,
        "total_mass_abs": total_mass_abs,
        "unassigned_fraction": unassigned_fraction,
    }

# Test on the same molecule used throughout the workflow
result = align_tokens_to_atoms(tokens, scores, offsets, spans)

print("Per-atom aligned scores (signed):")
for i, (s, e) in enumerate(spans):
    atom = mol.GetAtomWithIdx(i)
    print(f"  Atom {i:2d} ({atom.GetSymbol()}, chars[{s}:{e}]='{sample_smiles[s:e]}'): "
          f"{result['atom_scores'][i]:.4f}")

print(f"\nTotal mass (magnitude): {result['total_mass_abs']:.4f}")
print(f"Unassigned (syntax, magnitude): {result['syntax_mass_abs']:.4f}")
print(f"Unassigned (special tokens, magnitude): {result['special_token_mass_abs']:.4f}")
print(f"Fraction of total magnitude unassigned: {result['unassigned_fraction'] * 100:.1f}%")

Per-atom aligned scores (signed):
  Atom  0 (O, chars[0:1]='O'): -0.0924
  Atom  1 (N, chars[2:6]='[N+]'): -0.1535
  Atom  2 (O, chars[7:11]='[O-]'): 0.0486
  Atom  3 (C, chars[12:13]='c'): -0.0553
  Atom  4 (C, chars[14:15]='c'): 0.0585
  Atom  5 (C, chars[16:17]='c'): -0.0042
  Atom  6 (C, chars[18:19]='c'): -0.1525
  Atom  7 (C, chars[20:21]='c'): 0.0201
  Atom  8 (C, chars[21:22]='c'): 0.0201
  Atom  9 (C, chars[22:23]='c'): 0.0201
  Atom 10 (C, chars[24:25]='c'): 0.1306
  Atom 11 (C, chars[25:26]='c'): 0.1306
  Atom 12 (C, chars[26:27]='c'): 0.1306
  Atom 13 (C, chars[27:28]='c'): 0.1306
  Atom 14 (C, chars[29:30]='c'): 0.0882
  Atom 15 (C, chars[30:31]='c'): 0.0882
  Atom 16 (C, chars[31:32]='c'): 0.0882
  Atom 17 (C, chars[33:34]='c'): 0.0945
  Atom 18 (C, chars[35:36]='c'): -0.0491
  Atom 19 (C, chars[39:40]='C'): -0.0263
  Atom 20 (C, chars[40:41]='C'): -0.0263
  Atom 21 (C, chars[41:42]='C'): -0.0263
  Atom 22 (C, chars[42:43]='C'): -0.0263

Total mass (magnitude): 4.0661
Una

In [13]:
# We will need an unassigned-mass distribution check across a sample
# Combines extraction, atom spans, and alignment into one pipeline,run over thirty molecules to check if a specific percentage was typical or an outlier.

def full_chemberta_atom_pipeline(smiles, n_steps=50):
    """Processes a single molecule from start to finish through token attribution, atom spans, and alignment."""
    tokens, scores, offsets, delta = get_chemberta_token_attributions(smiles, n_steps=n_steps)
    mol = Chem.MolFromSmiles(smiles)
    spans = get_atom_char_spans(smiles)
    if mol.GetNumAtoms() != len(spans):
        return None  # Flag mismatches to avoid proceeding with incorrect atom counts
    result = align_tokens_to_atoms(tokens, scores, offsets, spans)
    result["convergence_delta"] = delta
    result["num_atoms"] = mol.GetNumAtoms()
    result["num_rings"] = mol.GetRingInfo().NumRings()
    return result

import random
random.seed(42)
sample_smiles_list = random.sample(list(df_clean[df_clean['split'] == 'test']['Drug_standardized']), 30)

unassigned_fractions = []
ring_counts = []
atom_counts = []
mismatches = 0
nan_count = 0

for smi in sample_smiles_list:
    r = full_chemberta_atom_pipeline(smi)
    if r is None:
        mismatches += 1
        continue
    frac = r["unassigned_fraction"]
    if np.isnan(frac):
        nan_count += 1
        continue
    unassigned_fractions.append(frac)
    ring_counts.append(r["num_rings"])
    atom_counts.append(r["num_atoms"])

import numpy as np
uf = np.array(unassigned_fractions)
print(f"Sampled {len(sample_smiles_list)} molecules, with {mismatches} atom count mismatches and "
      f"{nan_count} near zero total cases excluded from the statistics below.")
print(f"\nUnassigned mass fraction statistics (magnitude based and stable):")
print(f"Mean: {uf.mean() * 100:.1f}%, median: {np.median(uf) * 100:.1f}%, "
      f"min: {uf.min() * 100:.1f}%, max: {uf.max() * 100:.1f}%, std: {uf.std() * 100:.1f}%")

# Checking correlation with ring count and molecule size
print(f"\nComparison: our single example featured 5 rings and 23 atoms.")
print(f"Sample average: {np.mean(ring_counts):.1f} rings, {np.mean(atom_counts):.1f} atoms.")

correlation = np.corrcoef(ring_counts, uf)[0, 1]
print(f"Correlation between ring count and unassigned fraction: {correlation:.3f}")

Sampled 30 molecules, with 0 atom count mismatches and 0 near zero total cases excluded from the statistics below.

Unassigned mass fraction statistics (magnitude based and stable):
Mean: 45.8%, median: 45.9%, min: 19.4%, max: 64.1%, std: 9.6%

Comparison: our single example featured 5 rings and 23 atoms.
Sample average: 2.7 rings, 19.1 atoms.
Correlation between ring count and unassigned fraction: 0.227


#### ChemBERTa Full Test Set Attribution Extraction

In [14]:
test_df_ordered_cb = df_clean[df_clean['split'] == 'test'].reset_index(drop=True)

chemberta_results = {}  # Drug_ID -> {"atom_scores": ..., "unassigned_fraction": ..., "num_atoms": ...}
failed = 0
mismatches = 0

for i, row in test_df_ordered_cb.iterrows():
    smi = row['Drug_standardized']
    try:
        r = full_chemberta_atom_pipeline(smi)
        if r is None:
            mismatches += 1
            continue
        chemberta_results[row['Drug_ID']] = {
            "atom_scores": r["atom_scores"],
            "unassigned_fraction": r["unassigned_fraction"],
            "num_atoms": r["num_atoms"],
            "num_rings": r["num_rings"],
            "convergence_delta": r["convergence_delta"],
        }
    except Exception as e:
        failed += 1

    if (i + 1) % 100 == 0 or (i + 1) == len(test_df_ordered_cb):
        print(f"  Processed {i + 1} / {len(test_df_ordered_cb)}  "
              f"({len(chemberta_results)} succeeded, {mismatches} atom-count mismatches, {failed} other failures)")

print(f"\nExtracted ChemBERTa atom-level attributions for "
      f"{len(chemberta_results)} / {len(test_df_ordered_cb)} test molecules.")

# --- Full-set unassigned-fraction distribution (not just the 30-sample) ---
all_fracs = np.array([v["unassigned_fraction"] for v in chemberta_results.values()
                       if not np.isnan(v["unassigned_fraction"])])
print(f"\nFull-set unassigned-mass fraction -- mean: {all_fracs.mean()*100:.1f}%, "
      f"median: {np.median(all_fracs)*100:.1f}%, std: {all_fracs.std()*100:.1f}%")

with open("/kaggle/working/chemberta_attributions.pkl", "wb") as f:
    pickle.dump(chemberta_results, f)
print("Saved to /kaggle/working/chemberta_attributions.pkl")

  Processed 100 / 1457  (100 succeeded, 0 atom-count mismatches, 0 other failures)
  Processed 200 / 1457  (200 succeeded, 0 atom-count mismatches, 0 other failures)
  Processed 300 / 1457  (300 succeeded, 0 atom-count mismatches, 0 other failures)
  Processed 400 / 1457  (400 succeeded, 0 atom-count mismatches, 0 other failures)
  Processed 500 / 1457  (500 succeeded, 0 atom-count mismatches, 0 other failures)
  Processed 600 / 1457  (600 succeeded, 0 atom-count mismatches, 0 other failures)
  Processed 700 / 1457  (700 succeeded, 0 atom-count mismatches, 0 other failures)
  Processed 800 / 1457  (800 succeeded, 0 atom-count mismatches, 0 other failures)
  Processed 900 / 1457  (900 succeeded, 0 atom-count mismatches, 0 other failures)
  Processed 1000 / 1457  (1000 succeeded, 0 atom-count mismatches, 0 other failures)
  Processed 1100 / 1457  (1100 succeeded, 0 atom-count mismatches, 0 other failures)
  Processed 1200 / 1457  (1200 succeeded, 0 atom-count mismatches, 0 other failures

# Phase 5A - Masking Primitives

In [15]:
# (Verified edge cases: empty mask = identity, full mask = baseline, partial mask = exact rows/tokens affected)

# --- GIN: zero-feature soft masking ---
def mask_atoms_gin(x, atom_indices_to_mask):
    """Returns a copy of x with specified atom rows zeroed out."""
    x_masked = x.clone()
    if len(atom_indices_to_mask) > 0:
        x_masked[list(atom_indices_to_mask), :] = 0.0
    return x_masked

def predict_gin_masked(data, atom_indices_to_mask, device):
    """Evaluates GIN model with specified atoms masked."""
    data = data.to(device)
    x_masked = mask_atoms_gin(data.x, atom_indices_to_mask)
    batch = torch.zeros(x_masked.size(0), dtype=torch.long, device=device)
    with torch.no_grad():
        return gin_model(x_masked, data.edge_index, batch).item()

# --- ChemBERTa: token-overlap soft masking ---
def get_tokens_to_mask(atom_indices_to_mask, atom_spans, token_offsets):
    """
    Maps atom indices to token positions to mask based on character-span overlap.
    Also tracks 'collateral atoms' that share a token with the target atoms.
    """
    target_ranges = [atom_spans[i] for i in atom_indices_to_mask]

    def overlaps(a, b):
        return a[0] < b[1] and b[0] < a[1]

    token_positions = set()
    for tok_idx, (t_start, t_end) in enumerate(token_offsets):
        if t_end <= t_start:
            continue  # Skip special tokens
        if any(overlaps((t_start, t_end), target) for target in target_ranges):
            token_positions.add(tok_idx)

    collateral_atoms = set()
    for tok_idx in token_positions:
        t_start, t_end = token_offsets[tok_idx]
        for atom_idx, (a_start, a_end) in enumerate(atom_spans):
            if overlaps((t_start, t_end), (a_start, a_end)):
                collateral_atoms.add(atom_idx)
    collateral_atoms -= set(atom_indices_to_mask)

    return sorted(token_positions), sorted(collateral_atoms)

def predict_chemberta_masked(smiles, atom_indices_to_mask, device, max_length=128):
    """Evaluates ChemBERTa model with tokens overlapping target atoms replaced by PAD."""
    encoded = tokenizer(smiles, return_tensors="pt", return_offsets_mapping=True,
                        truncation=True, max_length=max_length)
    offsets = encoded.pop("offset_mapping")[0].tolist()
    input_ids = encoded["input_ids"].to(device)
    attention_mask = encoded["attention_mask"].to(device)

    real_len = int(attention_mask.sum().item())
    real_offsets = offsets[:real_len]

    spans = get_atom_char_spans(smiles)
    tok_positions, collateral = get_tokens_to_mask(atom_indices_to_mask, spans, real_offsets)

    input_ids_masked = input_ids.clone()
    for pos in tok_positions:
        input_ids_masked[0, pos] = tokenizer.pad_token_id

    with torch.no_grad():
        logits = chemberta_model(input_ids=input_ids_masked, attention_mask=attention_mask).logits
    return logits[0, 1].item(), collateral

# --- Edge-case verification on the sample molecule ---
print("=== GIN edge cases ===")
orig_pred = predict_gin_masked(sample_data, [], device)
all_masked_pred = predict_gin_masked(sample_data, list(range(sample_data.x.size(0))), device)
print(f"Mask nothing -> {orig_pred:.4f}  (compare to original unmasked prediction)")
print(f"Mask everything -> {all_masked_pred:.4f}  (compare to zero-baseline prediction)")

print("\n=== ChemBERTa edge cases ===")
n_atoms = len(get_atom_char_spans(sample_smiles))
orig_pred_cb, _ = predict_chemberta_masked(sample_smiles, [], device)
all_masked_cb, _ = predict_chemberta_masked(sample_smiles, list(range(n_atoms)), device)
print(f"Mask nothing -> {orig_pred_cb:.4f}  (compare to original unmasked prediction)")
print(f"Mask everything -> {all_masked_cb:.4f}  (compare to all-PAD baseline)")

=== GIN edge cases ===
Mask nothing -> 2.3848  (compare to original unmasked prediction)
Mask everything -> -83.1095  (compare to zero-baseline prediction)

=== ChemBERTa edge cases ===
Mask nothing -> 1.1690  (compare to original unmasked prediction)
Mask everything -> 0.8174  (compare to all-PAD baseline)


#### Exact True Baseline Checks for Evaluation in Drops

In [16]:
import torch

def get_true_baseline_prediction(smiles, max_length=128):
    """Computes the exact all-PAD baseline (retaining <s> and </s> tokens)."""
    encoded = tokenizer(smiles, return_tensors="pt", truncation=True, max_length=max_length)
    input_ids = encoded["input_ids"].to(device)
    attention_mask = encoded["attention_mask"].to(device)
    real_len = int(attention_mask.sum().item())

    baseline_ids = torch.full_like(input_ids, tokenizer.pad_token_id)
    baseline_ids[0, 0] = input_ids[0, 0]                        # Keep <s>
    baseline_ids[0, real_len - 1] = input_ids[0, real_len - 1]  # Keep </s>

    with torch.no_grad():
        logits = chemberta_model(input_ids=baseline_ids, attention_mask=attention_mask).logits
    return logits[0, 1].item()

true_baseline = get_true_baseline_prediction(sample_smiles)
original_pred = predict_chemberta_masked(sample_smiles, [], device)[0]
atom_mask_everything_pred = predict_chemberta_masked(
    sample_smiles, list(range(len(get_atom_char_spans(sample_smiles)))), device
)[0]

true_gap = original_pred - true_baseline
atom_mask_gap = original_pred - atom_mask_everything_pred
ceiling_fraction = atom_mask_gap / true_gap if abs(true_gap) > 1e-8 else float('nan')

print(f"Original prediction:              {original_pred:.4f}")
print(f"TRUE all-PAD baseline (exact):     {true_baseline:.4f}")
print(f"'Mask everything (atoms)' result:  {atom_mask_everything_pred:.4f}")
print(f"\nTrue gap (original - true baseline):        {true_gap:.4f}")
print(f"Atom-mask gap (original - atom-mask-all):   {atom_mask_gap:.4f}")
print(f"Atom-masking reaches {ceiling_fraction * 100:.1f}% of the true gap "
      f"(the rest is structurally unreachable via atom-only masking)")

Original prediction:              1.1690
TRUE all-PAD baseline (exact):     -0.1143
'Mask everything (atoms)' result:  0.8174

True gap (original - true baseline):        1.2833
Atom-mask gap (original - atom-mask-all):   0.3515
Atom-masking reaches 27.4% of the true gap (the rest is structurally unreachable via atom-only masking)
